# Voltametria Cíclica Reversível

As equações governantes são as mesmas da voltametria de varredura linear reversível — segunda lei de Fick adimensionalizada com equilíbrio de Nernst na superfície do eletrodo (eqs. 1.0 a 1.3 do modelo anterior).

A diferença está na forma de aplicação do potencial: após atingir o potencial de inversão $E_f$, a direção da varredura é invertida, permitindo que os processos de redução e oxidação sejam observados em um único experimento. O potencial varia com o tempo segundo:

$$E(\tau) = E_i + v\tau, \quad \text{se} \quad \tau \leq \tau_\lambda \tag{1.4a}$$

$$E(\tau) = E_f - v(\tau - \tau_\lambda), \quad \text{se} \quad \tau_\lambda \leq \tau \leq 2\tau_\lambda \tag{1.4b}$$

onde $v$ é a velocidade de varredura e $\tau_\lambda$ é o tempo de inversão, definido como:

$$\tau_\lambda = \frac{E_f - E_i}{v}$$

O tempo total do experimento é $2\tau_\lambda$, correspondendo à ida e volta da varredura.

## Bibliotecas:

In [ ]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Text, Button, HBox, VBox, Output
from IPython.display import display, clear_output

## Definindo o Modelo:

In [ ]:
model = pybamm.BaseModel()

concentration_o = pybamm.Variable("Concentração de O", domain="electrolyte")
concentration_r = pybamm.Variable("Concentração de R", domain="electrolyte")

initial_potential  = pybamm.Parameter("Potencial Inicial [V]")
final_potential    = pybamm.Parameter("Potencial Final [V]")
standard_potential = pybamm.Parameter("Potencial Padrão [V]")
scan_rate          = pybamm.Parameter("Velocidade de Varredura [V.s-1]")
faraday            = pybamm.Parameter("Constante de Faraday [C.mol-1]")
gas_constant       = pybamm.Parameter("Constante dos Gases [J.K-1.mol-1]")
temperature        = pybamm.Parameter("Temperatura [K]")

flux_o = -pybamm.grad(concentration_o)   # fluxo difusional de O
flux_r = -pybamm.grad(concentration_r)   # fluxo difusional de R

model.rhs = {
    concentration_o: -pybamm.div(flux_o),  # lei de Fick para O
    concentration_r: -pybamm.div(flux_r),  # lei de Fick para R
}

# condições iniciais
model.initial_conditions = {
    concentration_o: pybamm.Scalar(1),  # c_O(x,0) = 1: O uniforme e adimensionalizado
    concentration_r: pybamm.Scalar(0),  # c_R(x,0) = 0: R inicialmente ausente
}

# condições de contorno — Dirichlet
# esquerda (x=0): interface eletrodo/solução — equilíbrio de Nernst, potencial varia ciclicamente
# direita  (x=6): seio da solução            — difusão semi-infinita
nernst_factor     = faraday / (gas_constant * temperature)          # f = F/(RT)
total_time        = 2 * (final_potential - initial_potential) / scan_rate
cosine            = pybamm.cos((np.pi / total_time) * pybamm.t)
sine              = pybamm.sin((np.pi / total_time) * pybamm.t)
applied_potential = initial_potential + (2 * (final_potential - initial_potential) / np.pi) * pybamm.AbsoluteValue(pybamm.arctan(sine / cosine))
overpotential     = applied_potential - standard_potential
theta             = pybamm.exp(nernst_factor * overpotential)

model.boundary_conditions = {
    concentration_o: {
        "left":  (theta / (1 + theta), "Dirichlet"),  # c_O(0,t) = θ/(1+θ) — equilíbrio de Nernst
        "right": (pybamm.Scalar(1),    "Dirichlet"),  # c_O(∞,t) = 1       — difusão semi-infinita
    },
    concentration_r: {
        "left":  (1 / (1 + theta),    "Dirichlet"),  # c_R(0,t) = 1/(1+θ) — equilíbrio de Nernst
        "right": (pybamm.Scalar(0),    "Dirichlet"),  # c_R(∞,t) = 0       — difusão semi-infinita
    },
}

model.variables = {
    "Concentração de O":  concentration_o,
    "Concentração de R":  concentration_r,
    "Fluxo de O":         flux_o,
    "Fluxo de R":         flux_r,
    "Potencial Aplicado": applied_potential,
}

param = pybamm.ParameterValues(
    {
        "Potencial Inicial [V]":            "[input]",
        "Potencial Final [V]":              "[input]",
        "Potencial Padrão [V]":             "[input]",
        "Velocidade de Varredura [V.s-1]":  "[input]",
        "Constante de Faraday [C.mol-1]":   96485.3,
        "Constante dos Gases [J.K-1.mol-1]": 8.31446,
        "Temperatura [K]":                  298.15,
    }
)


# ── Geometria e Malha ─────────────────────────────────────────────────────────

x_variable = pybamm.SpatialVariable(
    "x", domain=["electrolyte"], coord_sys="cartesian"
)

geometry = {
    "electrolyte": {x_variable: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(6)}}
}

submesh_types = {
    "electrolyte": pybamm.MeshGenerator(
        pybamm.Exponential1DSubMesh,
        submesh_params={
            "side": "left",
            "stretch": 5,
        },
    )
}
variable_points = {x_variable: 400}
mesh            = pybamm.Mesh(geometry, submesh_types, variable_points)


# ── Discretização ─────────────────────────────────────────────────────────────

spatial_methods = {"electrolyte": pybamm.FiniteVolume()}
discretisation  = pybamm.Discretisation(mesh, spatial_methods)

param.process_model(model)
param.process_geometry(geometry)
discretisation.process_model(model)

## Gráfico Estático:

In [ ]:
static_solver = pybamm.ScipySolver()

# potenciais fixos para a varredura cíclica
DEFAULT_INITIAL_POTENTIAL  =  0.3   # potencial inicial [V]
DEFAULT_FINAL_POTENTIAL    = -0.3   # potencial final [V]
DEFAULT_STANDARD_POTENTIAL =  0     # potencial padrão [V]

scan_rates    = [-0.02, -0.05, -0.1, -0.2, -0.5]  # velocidades de varredura [V/s]
colors        = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

currents      = []
potentials    = []
peak_currents = []

for scan_rate_i in scan_rates:
    final_time = 2 * (DEFAULT_FINAL_POTENTIAL - DEFAULT_INITIAL_POTENTIAL) / scan_rate_i
    time       = np.linspace(0, final_time, 1000)

    solution = static_solver.solve(
        model, time,
        inputs={
            "Potencial Inicial [V]":           DEFAULT_INITIAL_POTENTIAL,
            "Potencial Final [V]":             DEFAULT_FINAL_POTENTIAL,
            "Potencial Padrão [V]":            DEFAULT_STANDARD_POTENTIAL,
            "Velocidade de Varredura [V.s-1]": scan_rate_i,
        }
    )

    flux_o_solution            = solution["Fluxo de O"]
    applied_potential_solution = solution["Potencial Aplicado"]

    current   = flux_o_solution(solution.t, x=0)
    potential = applied_potential_solution(solution.t)

    currents.append(current)
    potentials.append(potential)
    peak_currents.append(np.min(current))  # pico catódico (mínimo, pois a corrente é negativa)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for scan_rate_i, color, current, potential in zip(scan_rates, colors, currents, potentials):
    ax1.plot(potential, current, color=color, label=f"v = {scan_rate_i}")

ax1.set_xlabel(r"$E - E^0$ / V")
ax1.set_ylabel(r"$i$")
ax1.set_xlim([DEFAULT_FINAL_POTENTIAL, DEFAULT_INITIAL_POTENTIAL])
ax1.set_title("Voltamogramas para diferentes velocidades de varredura")
ax1.legend(loc="upper left")

scan_rate_roots = [np.sqrt(abs(scan_rate_i)) for scan_rate_i in scan_rates]
ax2.plot(scan_rate_roots, peak_currents, "o", color="tab:blue")
ax2.set_xlabel(r"$\sqrt{v}$")
ax2.set_ylabel(r"$i_p$")
ax2.set_title("Corrente de pico vs raiz da velocidade")

plt.tight_layout()
plt.show()

O gráfico da esquerda mostra os voltamogramas cíclicos para cinco velocidades de varredura diferentes. Cada curva apresenta dois picos: um **pico catódico** ($i_{pc}$), associado à redução de $O$ em $R$ na varredura direta, e um **pico anódico** ($i_{pa}$), associado à reoxidação de $R$ em $O$ na varredura inversa.

Para um sistema reversível, os dois picos são simétricos em amplitude e separados por uma diferença de potencial característica. A razão $|i_{pa}/i_{pc}| \approx 1$ e a separação entre os potenciais de pico $\Delta E_p \approx 59/n$ mV (a 25°C) são critérios diagnósticos de reversibilidade eletroquímica.

O gráfico da direita mostra que a corrente de pico cresce linearmente com $\sqrt{v}$, confirmando o controle difusional do processo — comportamento previsto pela equação de Randles-Ševčík:

$$i_p \propto \sqrt{v}$$

## Gráfico Interativo:

In [ ]:
interactive_solver = pybamm.ScipySolver()

output = Output()

def plot(
    initial_potential_1_str, final_potential_1_str,
    standard_potential_1_str, scan_rate_1_str,
    initial_potential_2_str, final_potential_2_str,
    standard_potential_2_str, scan_rate_2_str,
):
    with output:
        clear_output(wait=True)

        # validação das entradas
        try:
            initial_potential_1  = float(initial_potential_1_str)
            final_potential_1    = float(final_potential_1_str)
            standard_potential_1 = float(standard_potential_1_str)
            scan_rate_1          = float(scan_rate_1_str)
            initial_potential_2  = float(initial_potential_2_str)
            final_potential_2    = float(final_potential_2_str)
            standard_potential_2 = float(standard_potential_2_str)
            scan_rate_2          = float(scan_rate_2_str)
        except ValueError:
            print("Por favor, insira valores numéricos válidos.")
            return

        if scan_rate_1 == 0 or scan_rate_2 == 0:
            print("A velocidade de varredura não pode ser zero.")
            return

        # ── curva 1 ───────────────────────────────────────────────────────────
        final_time_1 = 2 * (final_potential_1 - initial_potential_1) / scan_rate_1
        solution_1   = interactive_solver.solve(
            model, np.linspace(0, final_time_1, 1000),
            inputs={
                "Potencial Inicial [V]":           initial_potential_1,
                "Potencial Final [V]":             final_potential_1,
                "Potencial Padrão [V]":            standard_potential_1,
                "Velocidade de Varredura [V.s-1]": scan_rate_1,
            }
        )
        current_1   = solution_1["Fluxo de O"](solution_1.t, x=0)
        potential_1 = solution_1["Potencial Aplicado"](solution_1.t)

        # ── curva 2 ───────────────────────────────────────────────────────────
        final_time_2 = 2 * (final_potential_2 - initial_potential_2) / scan_rate_2
        solution_2   = interactive_solver.solve(
            model, np.linspace(0, final_time_2, 1000),
            inputs={
                "Potencial Inicial [V]":           initial_potential_2,
                "Potencial Final [V]":             final_potential_2,
                "Potencial Padrão [V]":            standard_potential_2,
                "Velocidade de Varredura [V.s-1]": scan_rate_2,
            }
        )
        current_2   = solution_2["Fluxo de O"](solution_2.t, x=0)
        potential_2 = solution_2["Potencial Aplicado"](solution_2.t)

        # ── plotagem ──────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(7, 4))

        ax.plot(potential_1, current_1, color="tab:blue", linewidth=2, label="1ª Varredura")
        ax.plot(potential_2, current_2, color="tab:red",  linewidth=2, label="2ª Varredura")

        ax.set_xlim([
            min(final_potential_1, final_potential_2),
            max(initial_potential_1, initial_potential_2),
        ])
        ax.set_xlabel(r"$E$ / V")
        ax.set_ylabel(r"$i$")
        ax.set_title("Comparação de Voltamogramas")
        ax.legend()

        plt.tight_layout()
        display(fig)
        plt.close(fig)


# ── Widgets — 1ª Varredura ────────────────────────────────────────────────────

field_initial_potential_1  = Text(value="0.5",  description="$E_{i,1}$ [V]:",  style={"description_width": "initial"})
field_final_potential_1    = Text(value="-0.5", description="$E_{f,1}$ [V]:",  style={"description_width": "initial"})
field_standard_potential_1 = Text(value="0",    description="$E^0_1$ [V]:",    style={"description_width": "initial"})
field_scan_rate_1          = Text(value="-0.1", description="$v_1$ [V/s]:",    style={"description_width": "initial"})


# ── Widgets — 2ª Varredura ────────────────────────────────────────────────────

field_initial_potential_2  = Text(value="0.5",  description="$E_{i,2}$ [V]:",  style={"description_width": "initial"})
field_final_potential_2    = Text(value="-0.5", description="$E_{f,2}$ [V]:",  style={"description_width": "initial"})
field_standard_potential_2 = Text(value="0",    description="$E^0_2$ [V]:",    style={"description_width": "initial"})
field_scan_rate_2          = Text(value="-0.2", description="$v_2$ [V/s]:",    style={"description_width": "initial"})

button = Button(description="Recalcular", button_style="success")

def on_click(b):
    plot(
        field_initial_potential_1.value,  field_final_potential_1.value,
        field_standard_potential_1.value, field_scan_rate_1.value,
        field_initial_potential_2.value,  field_final_potential_2.value,
        field_standard_potential_2.value, field_scan_rate_2.value,
    )

button.on_click(on_click)

interface = VBox([
    HBox([field_initial_potential_1, field_final_potential_1, field_standard_potential_1, field_scan_rate_1]),
    HBox([field_initial_potential_2, field_final_potential_2, field_standard_potential_2, field_scan_rate_2]),
    button,
    output,
])
display(interface)

# exibe o gráfico inicial com os valores padrão
plot(
    field_initial_potential_1.value,  field_final_potential_1.value,
    field_standard_potential_1.value, field_scan_rate_1.value,
    field_initial_potential_2.value,  field_final_potential_2.value,
    field_standard_potential_2.value, field_scan_rate_2.value,
)